# Parallel Corpora Creation

This notebook creates parallel corpora for specific language pairs needed for similarity transfer experiments.

## Language Pairs

| Pair | Role | Languages |
|------|------|-----------|
| en–tl | baseline + distant | English, Tagalog |
| bik–tl | similar donor | Bikolano, Tagalog |
| en–hil | baseline + distant | English, Ilonggo (Hiligaynon) |
| msb–hil | similar donor | Masbatenyo, Ilonggo (Hiligaynon) |
| en–war | baseline + distant | English, Waray |
| hil–war | similar donor | Ilonggo (Hiligaynon), Waray |

## Imports

In [22]:
import json
from pathlib import Path
import pandas as pd

## Configuration

Define language pairs and mappings.

In [23]:
# Language code mappings
lang_code_to_name = {
    "en": "english",
    "tl": "tagalog",
    "bik": "bikolano",
    "hil": "ilonggo",
    "msb": "masbatenyo",
    "war": "waray"
}

# Define language pairs with their roles
language_pairs = [
    {
        "pair": "en-tl",
        "role": "baseline + distant",
        "src_code": "en",
        "tgt_code": "tl",
        "src_lang": "english",
        "tgt_lang": "tagalog"
    },
    {
        "pair": "bik-tl",
        "role": "similar donor",
        "src_code": "bik",
        "tgt_code": "tl",
        "src_lang": "bikolano",
        "tgt_lang": "tagalog"
    },
    {
        "pair": "en-hil",
        "role": "baseline + distant",
        "src_code": "en",
        "tgt_code": "hil",
        "src_lang": "english",
        "tgt_lang": "ilonggo"
    },
    {
        "pair": "msb-hil",
        "role": "similar donor",
        "src_code": "msb",
        "tgt_code": "hil",
        "src_lang": "masbatenyo",
        "tgt_lang": "ilonggo"
    },
    {
        "pair": "en-war",
        "role": "baseline + distant",
        "src_code": "en",
        "tgt_code": "war",
        "src_lang": "english",
        "tgt_lang": "waray"
    },
    {
        "pair": "hil-war",
        "role": "similar donor",
        "src_code": "hil",
        "tgt_code": "war",
        "src_lang": "ilonggo",
        "tgt_lang": "waray"
    }
]

print(f"Total language pairs: {len(language_pairs)}")
for pair_info in language_pairs:
    print(f"  {pair_info['pair']:10} | {pair_info['role']:20} | {pair_info['src_lang']:12} → {pair_info['tgt_lang']}")

Total language pairs: 6
  en-tl      | baseline + distant   | english      → tagalog
  bik-tl     | similar donor        | bikolano     → tagalog
  en-hil     | baseline + distant   | english      → ilonggo
  msb-hil    | similar donor        | masbatenyo   → ilonggo
  en-war     | baseline + distant   | english      → waray
  hil-war    | similar donor        | ilonggo      → waray


## Load Processed Data

Load the cleaned data from the preprocessing step.

In [24]:
processed_dir = Path("../data/processed")

# Load all language data
language_data = {}
for lang_name in ["english", "tagalog", "bikolano", "ilonggo", "masbatenyo", "waray"]:
    json_file = processed_dir / f"{lang_name}_clean.json"
    with open(json_file, "r", encoding="utf-8") as f:
        language_data[lang_name] = json.load(f)
    print(f"Loaded {lang_name}: {len(language_data[lang_name])} verses")

print(f"\nTotal languages loaded: {len(language_data)}")

Loaded english: 2948 verses
Loaded tagalog: 2948 verses
Loaded bikolano: 2948 verses
Loaded ilonggo: 2948 verses
Loaded masbatenyo: 2948 verses
Loaded waray: 2948 verses

Total languages loaded: 6


## Create Parallel Corpora

Generate parallel text files for each language pair.

In [25]:
def create_parallel_corpus(src_data, tgt_data, pair_name, output_dir):
    """
    Create parallel corpus files for a language pair.
    
    Args:
        src_data: List of verse dictionaries for source language
        tgt_data: List of verse dictionaries for target language
        pair_name: Name of the language pair (e.g., 'en-tl')
        output_dir: Directory to save the parallel corpus files
    
    Returns:
        Dictionary with statistics about the created corpus
    """
    # Verify that both datasets have the same length and alignment
    if len(src_data) != len(tgt_data):
        raise ValueError(f"Data length mismatch: {len(src_data)} vs {len(tgt_data)}")
    
    # Create output directory
    pair_dir = output_dir / pair_name
    pair_dir.mkdir(parents=True, exist_ok=True)
    
    # Extract language codes from pair name
    src_code, tgt_code = pair_name.split("-")
    
    # Write source file
    src_file = pair_dir / f"{pair_name}.{src_code}"
    with open(src_file, "w", encoding="utf-8") as f:
        for verse in src_data:
            f.write(f"{verse['text']}\n")
    
    # Write target file
    tgt_file = pair_dir / f"{pair_name}.{tgt_code}"
    with open(tgt_file, "w", encoding="utf-8") as f:
        for verse in tgt_data:
            f.write(f"{verse['text']}\n")
    
    # Write reference file (for traceability)
    ref_file = pair_dir / f"{pair_name}.ref"
    with open(ref_file, "w", encoding="utf-8") as f:
        for verse in src_data:
            f.write(f"{verse['reference']}\n")
    
    # Calculate statistics
    stats = {
        "pair": pair_name,
        "num_lines": len(src_data),
        "src_words": sum(len(v['text'].split()) for v in src_data),
        "tgt_words": sum(len(v['text'].split()) for v in tgt_data),
        "src_file": str(src_file),
        "tgt_file": str(tgt_file),
        "ref_file": str(ref_file)
    }
    
    return stats

# Create output directory
parallel_dir = Path("../data/parallel")
parallel_dir.mkdir(parents=True, exist_ok=True)

# Create parallel corpora for all language pairs
corpus_stats = []

print("Creating parallel corpora...")
print("=" * 80)

for pair_info in language_pairs:
    pair_name = pair_info["pair"]
    src_lang = pair_info["src_lang"]
    tgt_lang = pair_info["tgt_lang"]
    
    print(f"\nCreating {pair_name} ({src_lang} → {tgt_lang})...")
    
    src_data = language_data[src_lang]
    tgt_data = language_data[tgt_lang]
    
    stats = create_parallel_corpus(src_data, tgt_data, pair_name, parallel_dir)
    corpus_stats.append({**pair_info, **stats})
    
    print(f"  ✓ Created {stats['num_lines']} parallel sentences")
    print(f"  ✓ Source words: {stats['src_words']:,}")
    print(f"  ✓ Target words: {stats['tgt_words']:,}")

print("\n" + "=" * 80)
print("✓ All parallel corpora created successfully!")
print("=" * 80)

Creating parallel corpora...

Creating en-tl (english → tagalog)...
  ✓ Created 2948 parallel sentences
  ✓ Source words: 62,810
  ✓ Target words: 62,619

Creating bik-tl (bikolano → tagalog)...
  ✓ Created 2948 parallel sentences
  ✓ Source words: 57,918
  ✓ Target words: 62,619

Creating en-hil (english → ilonggo)...
  ✓ Created 2948 parallel sentences
  ✓ Source words: 62,810
  ✓ Target words: 67,835

Creating msb-hil (masbatenyo → ilonggo)...
  ✓ Created 2948 parallel sentences
  ✓ Source words: 65,377
  ✓ Target words: 67,835

Creating en-war (english → waray)...
  ✓ Created 2948 parallel sentences
  ✓ Source words: 62,810
  ✓ Target words: 65,094

Creating hil-war (ilonggo → waray)...
  ✓ Created 2948 parallel sentences
  ✓ Source words: 67,835
  ✓ Target words: 65,094

✓ All parallel corpora created successfully!


## Summary Statistics

Display comprehensive statistics about all created parallel corpora.

In [26]:
# Create summary DataFrame
df_stats = pd.DataFrame(corpus_stats)

print("\n" + "=" * 100)
print("PARALLEL CORPORA SUMMARY")
print("=" * 100)

print("\nBy Language Pair:")
print("-" * 100)
for _, row in df_stats.iterrows():
    print(f"\n{row['pair'].upper()} ({row['role']})")
    print(f"  Languages: {row['src_lang'].capitalize()} → {row['tgt_lang'].capitalize()}")
    print(f"  Sentences: {row['num_lines']:,}")
    print(f"  Source words: {row['src_words']:,} | Avg per sentence: {row['src_words']/row['num_lines']:.1f}")
    print(f"  Target words: {row['tgt_words']:,} | Avg per sentence: {row['tgt_words']/row['num_lines']:.1f}")

print("\n" + "-" * 100)
print("\nBy Role:")
print("-" * 100)

baseline_pairs = df_stats[df_stats['role'] == 'baseline + distant']
donor_pairs = df_stats[df_stats['role'] == 'similar donor']

print(f"\nBaseline + Distant pairs ({len(baseline_pairs)}):")
for _, row in baseline_pairs.iterrows():
    print(f"  • {row['pair']:10} | {row['src_lang']:12} → {row['tgt_lang']:12} | {row['num_lines']:,} sentences")

print(f"\nSimilar Donor pairs ({len(donor_pairs)}):")
for _, row in donor_pairs.iterrows():
    print(f"  • {row['pair']:10} | {row['src_lang']:12} → {row['tgt_lang']:12} | {row['num_lines']:,} sentences")

print("\n" + "=" * 100)
print(f"Total parallel corpora: {len(corpus_stats)}")
print(f"Total sentences per corpus: {df_stats['num_lines'].iloc[0]:,}")
print(f"Output directory: {parallel_dir.absolute()}")
print("=" * 100)


PARALLEL CORPORA SUMMARY

By Language Pair:
----------------------------------------------------------------------------------------------------

EN-TL (baseline + distant)
  Languages: English → Tagalog
  Sentences: 2,948
  Source words: 62,810 | Avg per sentence: 21.3
  Target words: 62,619 | Avg per sentence: 21.2

BIK-TL (similar donor)
  Languages: Bikolano → Tagalog
  Sentences: 2,948
  Source words: 57,918 | Avg per sentence: 19.6
  Target words: 62,619 | Avg per sentence: 21.2

EN-HIL (baseline + distant)
  Languages: English → Ilonggo
  Sentences: 2,948
  Source words: 62,810 | Avg per sentence: 21.3
  Target words: 67,835 | Avg per sentence: 23.0

MSB-HIL (similar donor)
  Languages: Masbatenyo → Ilonggo
  Sentences: 2,948
  Source words: 65,377 | Avg per sentence: 22.2
  Target words: 67,835 | Avg per sentence: 23.0

EN-WAR (baseline + distant)
  Languages: English → Waray
  Sentences: 2,948
  Source words: 62,810 | Avg per sentence: 21.3
  Target words: 65,094 | Avg per se

## Save Metadata

Save corpus metadata for future reference.

In [27]:
# Save metadata as JSON
# Convert numpy types to Python types for JSON serialization
corpus_stats_json = []
for stat in corpus_stats:
    stat_copy = stat.copy()
    stat_copy['num_lines'] = int(stat_copy['num_lines'])
    stat_copy['src_words'] = int(stat_copy['src_words'])
    stat_copy['tgt_words'] = int(stat_copy['tgt_words'])
    corpus_stats_json.append(stat_copy)

metadata = {
    "description": "Parallel corpora for Philippine low-resource language MT with similarity transfer",
    "creation_date": "2025-11-08",
    "source": "Bible (Matthew, Mark, Luke)",
    "total_pairs": len(corpus_stats_json),
    "sentences_per_corpus": int(df_stats['num_lines'].iloc[0]),
    "language_pairs": corpus_stats_json
}

metadata_file = parallel_dir / "corpus_metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"✓ Metadata saved to {metadata_file}")

# Save statistics as CSV
csv_file = parallel_dir / "corpus_statistics.csv"
df_stats[['pair', 'role', 'src_lang', 'tgt_lang', 'num_lines', 'src_words', 'tgt_words']].to_csv(
    csv_file, index=False
)

print(f"✓ Statistics saved to {csv_file}")

✓ Metadata saved to ..\data\parallel\corpus_metadata.json
✓ Statistics saved to ..\data\parallel\corpus_statistics.csv


## Create TSV/CSV Versions (Alternative Format)

Create TSV versions of the parallel corpora for frameworks that prefer this format.

In [28]:
# Create TSV and CSV versions for each language pair
print("Creating TSV/CSV versions...")
print("=" * 80)

for pair_info in language_pairs:
    pair_name = pair_info["pair"]
    src_lang = pair_info["src_lang"]
    tgt_lang = pair_info["tgt_lang"]
    src_code = pair_info["src_code"]
    tgt_code = pair_info["tgt_code"]
    
    pair_dir = parallel_dir / pair_name
    
    src_data = language_data[src_lang]
    tgt_data = language_data[tgt_lang]
    
    # Create TSV file (tab-separated)
    tsv_file = pair_dir / f"{pair_name}.tsv"
    with open(tsv_file, "w", encoding="utf-8") as f:
        # Write header
        f.write(f"{src_code}\t{tgt_code}\n")
        # Write data
        for src_verse, tgt_verse in zip(src_data, tgt_data):
            f.write(f"{src_verse['text']}\t{tgt_verse['text']}\n")
    
    # Create CSV file (comma-separated, with proper escaping)
    csv_file = pair_dir / f"{pair_name}.csv"
    df_pair = pd.DataFrame({
        src_code: [v['text'] for v in src_data],
        tgt_code: [v['text'] for v in tgt_data]
    })
    df_pair.to_csv(csv_file, index=False, encoding='utf-8')
    
    print(f"✓ Created {pair_name}.tsv and {pair_name}.csv")

print("\n" + "=" * 80)
print("✓ All TSV/CSV files created successfully!")
print("\n📁 Each language pair now has:")
print("  • Separate text files (.src, .tgt) - for fairseq, OpenNMT, Transformers")
print("  • TSV file (.tsv) - tab-separated with header")
print("  • CSV file (.csv) - comma-separated with proper escaping")
print("  • Reference file (.ref) - verse references for traceability")
print("=" * 80)

Creating TSV/CSV versions...
✓ Created en-tl.tsv and en-tl.csv
✓ Created bik-tl.tsv and bik-tl.csv
✓ Created en-hil.tsv and en-hil.csv
✓ Created msb-hil.tsv and msb-hil.csv
✓ Created en-war.tsv and en-war.csv
✓ Created hil-war.tsv and hil-war.csv

✓ All TSV/CSV files created successfully!

📁 Each language pair now has:
  • Separate text files (.src, .tgt) - for fairseq, OpenNMT, Transformers
  • TSV file (.tsv) - tab-separated with header
  • CSV file (.csv) - comma-separated with proper escaping
  • Reference file (.ref) - verse references for traceability


## Verify Alignment

Verify that parallel files are properly aligned by checking a sample.

In [29]:
# Verify alignment for a sample pair
sample_pair = language_pairs[0]  # en-tl
pair_name = sample_pair['pair']
src_code = sample_pair['src_code']
tgt_code = sample_pair['tgt_code']

pair_dir = parallel_dir / pair_name

# Read first 5 lines from each file
src_file = pair_dir / f"{pair_name}.{src_code}"
tgt_file = pair_dir / f"{pair_name}.{tgt_code}"
ref_file = pair_dir / f"{pair_name}.ref"

with open(src_file, "r", encoding="utf-8") as f:
    src_lines = [line.strip() for line in f.readlines()[:5]]

with open(tgt_file, "r", encoding="utf-8") as f:
    tgt_lines = [line.strip() for line in f.readlines()[:5]]

with open(ref_file, "r", encoding="utf-8") as f:
    ref_lines = [line.strip() for line in f.readlines()[:5]]

print(f"Alignment Verification for {pair_name.upper()}")
print("=" * 100)

for i, (ref, src, tgt) in enumerate(zip(ref_lines, src_lines, tgt_lines), 1):
    print(f"\nSample {i}: {ref}")
    print(f"  {src_code.upper()}: {src}")
    print(f"  {tgt_code.upper()}: {tgt}")

print("\n" + "=" * 100)
print("✓ Alignment verified successfully!")

Alignment Verification for EN-TL

Sample 1: LUK 1:1
  EN: Forasmuch as many have taken in hand to draw up a narrative concerning those matters which have been fulfilled among us,
  TL: Yamang marami ang nagsikap mag-ayos ng isang kasaysayan tungkol sa mga bagay na naganap sa gitna natin,

Sample 2: LUK 1:2
  EN: even as they delivered them unto us, who from the beginning were eyewitnesses and ministers of the word,
  TL: ayon sa ipinaalam sa atin ng mga taong buhat sa pasimula ay mga saksing nakakita at mga tagapangaral ng salita,

Sample 3: LUK 1:3
  EN: it seemed good to me also, having traced the course of all things accurately from the first, to write unto thee in order, most excellent Theophilus;
  TL: ay minabuti ko naman, pagkatapos na siyasating mabuti ang lahat ng mga pangyayari buhat sa pasimula, na sumulat ng isang maayos na salaysay para sa iyo, kagalang-galang na Teofilo,

Sample 4: LUK 1:4
  EN: that thou mightest know the certainty concerning the things wherein thou wast